<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set1/blob/main/DeepSeek_Set1_ZeroShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [57]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA NOT available. Using CPU.")

CUDA is available! Using GPU.
GPU device name: NVIDIA A100-SXM4-40GB


In [58]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       5.8Gi        46Gi        28Mi        30Gi        76Gi
Swap:             0B          0B          0B


In [59]:
from google.colab import files
import zipfile
import torch

import os
from transformers import AutoModelForCausalLM, AutoTokenizer

In [60]:
import shutil

# CLEANUP - Remove old folders before extraction
if os.path.exists('input_folder'):
    shutil.rmtree('input_folder')
if os.path.exists('output_folder'):
    shutil.rmtree('output_folder')


In [61]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving Testing.zip to Testing (2).zip
Files extracted to 'input_folder/'


In [62]:
# 3. Load model
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-instruct")
tokenizer.pad_token = tokenizer.eos_token  # Add this line


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [63]:
# BATCHING — DeepSeek Chat Template Version
def translate_batch(c_code_list):
    results = []

    for c_code in c_code_list:
        # Build user + system messages
        user_prompt = f"""
Translate this C code to C++ code:

C Code:
{c_code}

C++ Code:
"""
        system_prompt = """You are an expert code translator. Your ONLY task is to convert C code to C++ code.
Rules you MUST follow:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        # Apply DeepSeek chat template
        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True, padding = True,
        ).to(model.device)

        # Generate translation
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            top_p=0.9
        )

        # Extract and decode the new tokens only (after prompt)
        generated_ids = outputs[:, inputs.shape[1]:]
        decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        # Clean DeepSeek markdown or labels if present
        decoded = (decoded.replace("```cpp", "") .replace("```c++", "") .replace("```", "") .strip())


        # Clean and store
        translation = decoded.strip()
        results.append(translation)

        # Free memory (especially on Colab GPUs)
        import gc, torch
        gc.collect()
        torch.cuda.empty_cache()

    return results


In [64]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")

                import gc
                gc.collect()
                torch.cuda.empty_cache()
                # Clear batch lists
                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generati

Translated: input_folder/Testing/C/91.c → output_folder/Testing/C/91.cpp
Translated: input_folder/Testing/C/92.c → output_folder/Testing/C/92.cpp
Translated: input_folder/Testing/C/9099.c → output_folder/Testing/C/9099.cpp
Translated: input_folder/Testing/C/73.c → output_folder/Testing/C/73.cpp


In [65]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [66]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r output.zip output_folder
colab_files.download('output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
updating: output_folder/ (stored 0%)
updating: output_folder/Testing/ (stored 0%)
updating: output_folder/Testing/C/ (stored 0%)
updating: output_folder/Testing/C/92.cpp (deflated 69%)
updating: output_folder/Testing/C/91.cpp (deflated 65%)
updating: output_folder/Testing/C/9099.cpp (deflated 37%)
updating: output_folder/Testing/C/73.cpp (deflated 63%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.


In [67]:
print(model.generation_config)


GenerationConfig {
  "bos_token_id": 32013,
  "eos_token_id": 32021
}

